In [ ]:
import json
import os
import pandas as pd

In [ ]:
def extract_leg_keypoints_from_json(json_path):
    """Extract leg keypoints from a given OpenPose JSON file."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Define the leg keypoints indices
    leg_indices = {
        "Right Hip": 9,
        "Right Knee": 10,
        "Right Ankle": 11,
        "Left Hip": 12,
        "Left Knee": 13,
        "Left Ankle": 14
    }
    
    # Determine if there's one or two subjects
    num_subjects = len(data["people"])
    
    if num_subjects >= 2:
        keypoints_2d = data["people"][1]["pose_keypoints_2d"]
    elif num_subjects == 1:
        keypoints_2d = data["people"][0]["pose_keypoints_2d"]
    else:
        keypoints_2d = []
    
    # Extract the leg keypoints
    frame_keypoints = {}
    for leg_part, index in leg_indices.items():
        start_idx = index * 3
        frame_keypoints[f"{leg_part}_x"] = keypoints_2d[start_idx] if keypoints_2d else None
        frame_keypoints[f"{leg_part}_y"] = keypoints_2d[start_idx + 1] if keypoints_2d else None
        frame_keypoints[f"{leg_part}_confidence"] = keypoints_2d[start_idx + 2] if keypoints_2d else None
    
    return frame_keypoints


In [ ]:
def jsons_to_csv(directory_path, csv_output_path):
    """Convert a directory of JSON files to a single CSV file."""
    # Get a list of all JSON files in the directory
    json_files = [file for file in os.listdir(directory_path) if file.endswith('.json')]
    json_files.sort()

    # Extract keypoints from each JSON file
    keypoints_data = [extract_leg_keypoints_from_json(os.path.join(directory_path, file)) for file in json_files]

    # Convert to DataFrame and save to CSV
    df_keypoints = pd.DataFrame(keypoints_data)
    df_keypoints.to_csv(csv_output_path, index=False)


In [ ]:
input_directory = input("Enter the path to the directory containing JSON files: ")
output_csv_path = input("Enter the path where the CSV file should be saved: ")
jsons_to_csv(input_directory, output_csv_path)
print(f"Data saved to {output_csv_path}")
